## Vergleich globaler SHAP-Muster zwischen Top-38 und Stable-70

Dieses Skript vergleicht die **globalen SHAP-Ergebnisse** zweier Feature-Sets (`Top38` und `Stable70`) und prüft, wie konsistent sich deren wichtigste Modelltreiber überlappen. Dabei werden sowohl **einzelne Features** als auch **übergeordnete Domänen** betrachtet.

### Ablauf

1. Zunächst werden für beide Modelle Tabellen mit den absoluten SHAP-Werten erstellt.
2. Daraus werden pro Modell
   - Rangplätze
   - normalisierte SHAP-Anteile
   berechnet.
3. Anschließend werden die gemeinsamen Features beider Modelle zusammengeführt.

### Analysierte Vergleiche

Der Code erstellt mehrere Vergleichsebenen:

- **Globaler Vergleich**  
  Bestimmt die Überlappung der Top-5, Top-10 und Top-15 Features sowie die Spearman-Korrelation der Feature-Wichtigkeit und Ränge.

- **Bestätigungsanalyse**  
  Prüft, inwieweit die wichtigsten Stable70-Features auch im breiteren Top38-Modell wieder auftauchen, inklusive Rangdifferenzen und Verhältnis der normalisierten SHAP-Anteile.

- **Domain-Analyse**  
  Ordnet Features heuristisch in inhaltliche Gruppen ein, z. B.:
  - `ISO / Asymmetry`
  - `Unilateral CMJ`
  - `Global CMJ`
  - `KAI`
  - `Demographics`

  Danach wird verglichen, wie stark diese Domänen in beiden Modellen insgesamt vertreten sind.

### Ausgabe und Nutzen

Die Ergebnisse werden als formatierte HTML-Tabellen im Notebook angezeigt. Dadurch lässt sich kompakt beurteilen,

- wie ähnlich sich beide Modelle in ihren wichtigsten Features sind,
- welche Merkmale im stabileren Modell bestätigt werden,
- und ob sich auf **Domänenebene** ein konsistentes inhaltliches Muster zeigt.

In [7]:
top38_data = [
    ("INV_CMJ_uni_Peak braking force", 0.675567),
    ("ISO_Drehmoment_Seitenunterschied Extension relativ", 0.615490),
    ("UNINV_CMJ_uni_Peak Loading Force", 0.605353),
    ("ISO_Drehmoment_Seitenunterschied Extension absolut", 0.561944),
    ("UNINV_CMJ_uni_Av. propulsive force", 0.512865),
    ("UNINV_CMJ_uni_Peak Landing", 0.511161),
    ("INV_Arbeit_Extension", 0.505367),
    ("INV_Arbeit_Flexion", 0.497142),
    ("INV_CMJ_uni_Durchschnittliche Bremsgeschwindigkeit", 0.440862),
    ("UNINV_CMJ_uni_Braking Duration", 0.429501),
    ("CMJ_Peak loading force", 0.372600),
    ("UNINV_CMJ_uni_Max Rate of Force Development", 0.365671),
    ("ISO_Arbeit_Seitenunterschied Flexion absolut", 0.348931),
    ("INV_CMJ_uni_Peak Loading Force", 0.291478),
    ("CMJ_Jump Height impulse", 0.262919),
    ("CMJ_Countermovement depth", 0.256000),
    ("INV_ISO_Arbeit_Unterschied Extension Flexion", 0.252922),
    ("CMJ_Jump Height flighttime", 0.237659),
    ("INV_CMJ_uni_Av. propulsive force", 0.231594),
    ("UNINV_CMJ_uni_Bremsimpuls", 0.214771),
    ("INV_CMJ_uni_Net Impulse", 0.209930),
    ("CMJ_KAI ecc", 0.203972),
    ("UNINV_ISO_Drehmoment_Unterschied Extension Flexion", 0.201411),
    ("CMJ_Braking duration", 0.193422),
    ("CMJ_Vertical Stiffness", 0.186794),
    ("CMJ_Propulsive duration", 0.168377),
    ("UNINV_ISO_Max Flexion", 0.155349),
    ("UNINV_ISO_Drehmoment_Verhaeltnis Flexion Extension", 0.155183),
    ("UNINV_CMJ_uni_Peak Braking Force", 0.133416),
    ("INV_CMJ_uni_Durchschnittliche Bremsleistung", 0.122222),
    ("INV_ISO_Drehmoment_Unterschied Extension Flexion", 0.116909),
    ("CMJ_Peak landing force", 0.102338),
    ("INV_CMJ_uni_Vertical Stiffness", 0.089781),
    ("INV_ISO_Drehmoment_Verhaeltnis Flexion Extension", 0.064010),
    ("CMJ_KAI con", 0.060053),
    ("INV_ISO_Arbeit_Verhaeltnis Flexion Extension", 0.058132),
    ("Geschlecht_weiblich", 0.052881),
    ("INV_CMJ_uni_Max Rate of Force Development", 0.048435),
]

stable70_data = [
    ("INV_CMJ_uni_Peak braking force", 1.314807),
    ("UNINV_CMJ_uni_Av. propulsive force", 1.034452),
    ("INV_Arbeit_Flexion", 0.850963),
    ("UNINV_CMJ_uni_Peak Landing Force", 0.544977),
    ("INV_ISO_Drehmoment_Verhaeltnis Flexion Extension", 0.419216),
    ("ISO_Arbeit_Seitenunterschied Flexion absolut", 0.357352),
    ("CMJ_KAI ecc", 0.325351),
    ("UNINV_CMJ_uni_Bremsimpuls", 0.268829),
    ("INV_ISO_Arbeit_Unterschied Extension Flexion", 0.267264),
    ("CMJ_KAI con", 0.260830),
    ("Geschlecht_weiblich", 0.162247),
    ("INV_ISO_Drehmoment_Unterschied Extension Flexion", 0.153571),
    ("UNINV_ISO_Max Flexion", 0.134282),
    ("INV_CMJ_uni_Max Rate of Force Development", 0.134039),
    ("INV_ISO_Arbeit_Verhaeltnis Flexion Extension", 0.073297),
]

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from IPython.display import display, HTML

# ===============================
# 1) DATAFRAMES
# ===============================

df_top38 = pd.DataFrame(top38_data, columns=["feature", "abs_shap_Top38"])
df_stable70 = pd.DataFrame(stable70_data, columns=["feature", "abs_shap_Stable70"])

# Ranking
df_top38["rank_Top38"] = df_top38["abs_shap_Top38"].rank(ascending=False, method="min").astype(int)
df_stable70["rank_Stable70"] = df_stable70["abs_shap_Stable70"].rank(ascending=False, method="min").astype(int)

# Normalisierung
df_top38["norm_shap_Top38"] = df_top38["abs_shap_Top38"] / df_top38["abs_shap_Top38"].sum()
df_stable70["norm_shap_Stable70"] = df_stable70["abs_shap_Stable70"] / df_stable70["abs_shap_Stable70"].sum()

# Merge
df_compare = df_top38.merge(df_stable70, on="feature", how="inner")

# ===============================
# 2) HYPOTHESE 1 – GLOBAL
# ===============================

def top_overlap(df1, df2, n):
    top1 = set(df1.nsmallest(n, "rank_Top38")["feature"])
    top2 = set(df2.nsmallest(n, "rank_Stable70")["feature"])
    return len(top1 & top2), len(top1 & top2) / n

overlap5 = top_overlap(df_top38, df_stable70, 5)
overlap10 = top_overlap(df_top38, df_stable70, 10)
overlap15 = top_overlap(df_top38, df_stable70, 15)

rho_abs, _ = spearmanr(df_compare["abs_shap_Top38"], df_compare["abs_shap_Stable70"])
rho_rank, _ = spearmanr(df_compare["rank_Top38"], df_compare["rank_Stable70"])

table_A = pd.DataFrame({
    "Metric": [
        "Common features",
        "Top-5 overlap (%)",
        "Top-10 overlap (%)",
        "Top-15 overlap (%)",
        "Spearman rho (importance)",
        "Spearman rho (rank)",
    ],
    "Value": [
        len(df_compare),
        overlap5[1],
        overlap10[1],
        overlap15[1],
        rho_abs,
        rho_rank,
    ]
})

# ===============================
# 3) HYPOTHESE 2 – BESTÄTIGUNG
# ===============================

df_confirm = df_compare.copy()
df_confirm["rank_diff"] = df_confirm["rank_Top38"] - df_confirm["rank_Stable70"]
df_confirm["norm_ratio"] = df_confirm["norm_shap_Stable70"] / df_confirm["norm_shap_Top38"]

# Summary
top10_stable = set(df_stable70.nsmallest(10, "rank_Stable70")["feature"])
top15_top38 = set(df_top38.nsmallest(15, "rank_Top38")["feature"])

confirmation_rate = len(top10_stable & top15_top38) / len(top10_stable)

table_B_summary = pd.DataFrame({
    "Metric": [
        "Stable70 Top-10 in Top38 Top-15 (%)",
        "Median rank in Top38",
        "Median norm_SHAP Top38",
        "Median norm_SHAP Stable70"
    ],
    "Value": [
        confirmation_rate,
        df_confirm["rank_Top38"].median(),
        df_confirm["norm_shap_Top38"].median(),
        df_confirm["norm_shap_Stable70"].median()
    ]
})

# ===============================
# 4) DOMAIN ANALYSIS
# ===============================

def assign_domain(f):
    f = f.lower()
    if "iso" in f or "arbeit" in f:
        return "ISO / Asymmetry"
    if "cmj_uni" in f:
        return "Unilateral CMJ"
    if "cmj_" in f:
        return "Global CMJ"
    if "kai" in f:
        return "KAI"
    if "geschlecht" in f:
        return "Demographics"
    return "Other"

df_top38["domain"] = df_top38["feature"].apply(assign_domain)
df_stable70["domain"] = df_stable70["feature"].apply(assign_domain)

domain_top38 = df_top38.groupby("domain")["norm_shap_Top38"].sum()
domain_stable70 = df_stable70.groupby("domain")["norm_shap_Stable70"].sum()

table_D = pd.concat([domain_top38, domain_stable70], axis=1).fillna(0)
table_D.columns = ["Top38", "Stable70"]

# ===============================
# 5) OUTPUT (nur grafisch angepasst)
# ===============================

def _format_for_html(df, decimals=3):
    df_out = df.copy()
    for col in df_out.columns:
        if pd.api.types.is_float_dtype(df_out[col]):
            df_out[col] = df_out[col].map(lambda x: f"{x:.{decimals}f}")
    return df_out

def _show_html_table(title, df, decimals=3):
    df_show = _format_for_html(df, decimals=decimals)
    html = f"""
    <div style="margin: 18px 0 28px 0;">
        <h3 style="margin-bottom: 10px;">{title}</h3>
        {df_show.to_html(index=False, escape=False)}
    </div>
    """
    display(HTML(html))

_show_html_table("TABELLE A: GLOBALER VERGLEICH", table_A, decimals=3)
_show_html_table("TABELLE B: BESTÄTIGUNG (Stable70 → Top38)", table_B_summary, decimals=3)
_show_html_table("DETAIL: FEATURE-BESTÄTIGUNG", df_confirm.sort_values("rank_Stable70"), decimals=3)
_show_html_table("DOMAIN VERGLEICH", table_D.sort_values("Stable70", ascending=False).reset_index(), decimals=3)

Metric,Value
Common features,14.000
Top-5 overlap (%),0.400
Top-10 overlap (%),0.300
Top-15 overlap (%),0.267
Spearman rho (importance),0.811
Spearman rho (rank),0.811


Metric,Value
Stable70 Top-10 in Top38 Top-15 (%),0.400
Median rank in Top38,24.500
Median norm_SHAP Top38,0.017
Median norm_SHAP Stable70,0.043


feature,abs_shap_Top38,rank_Top38,norm_shap_Top38,abs_shap_Stable70,rank_Stable70,norm_shap_Stable70,rank_diff,norm_ratio
INV_CMJ_uni_Peak braking force,0.676,1,0.064,1.315,1,0.209,0,3.247
UNINV_CMJ_uni_Av. propulsive force,0.513,5,0.049,1.034,2,0.164,3,3.365
INV_Arbeit_Flexion,0.497,8,0.047,0.851,3,0.135,5,2.856
INV_ISO_Drehmoment_Verhaeltnis Flexion Extension,0.064,34,0.006,0.419,5,0.067,29,10.926
ISO_Arbeit_Seitenunterschied Flexion absolut,0.349,13,0.033,0.357,6,0.057,7,1.709
CMJ_KAI ecc,0.204,22,0.019,0.325,7,0.052,15,2.661
UNINV_CMJ_uni_Bremsimpuls,0.215,20,0.020,0.269,8,0.043,12,2.088
INV_ISO_Arbeit_Unterschied Extension Flexion,0.253,17,0.024,0.267,9,0.042,8,1.763
CMJ_KAI con,0.060,35,0.006,0.261,10,0.041,25,7.246
Geschlecht_weiblich,0.053,37,0.005,0.162,11,0.026,26,5.118


domain,Top38,Stable70
Unilateral CMJ,0.464,0.523
ISO / Asymmetry,0.336,0.358
Global CMJ,0.194,0.093
Demographics,0.005,0.026
